# PSPE on Colab — real backbones, real data

Runs the parts of PSPE that need a GPU: **real open-weight backbones** on
**real Sentinel-2 imagery** (EuroSAT → NDVI field reconstruction), plus a real
LM for the explanation head.

**Perceive backbone: a standard-ViT open-weight vision tower** — SigLIP by
default (the vision half of many VLMs), or CLIP / DINOv2. These take
`pixel_values` and run with LoRA on a free T4.

> Note on Qwen2-VL / moondream2: their vision towers use *packed dynamic-
> resolution patches* and require a `grid_thw` argument from the model's image
> processor — they cannot be driven as a plain image encoder, and the module
> raises a clear error if you pass one. Use a standard-ViT vision backbone for
> the field-regression task. (The *language* head below still uses Qwen2.5,
> which has the standard causal-LM interface and works directly.)

Everything is free: EuroSAT is on the HuggingFace Hub (no Copernicus account),
the backbones are Apache-2.0.

Runtime → Change runtime type → **T4 GPU** before running.

Produces what the local runs cannot:
- perception numbers with `backbone_is_stub: false`
- field-reconstruction error on **real satellite imagery**
- the two missing perception baselines (zero-shot vision, CNN/ViT regression)
- non-degenerate explanation briefs from a real LM

## 1. Environment

In [ ]:
import torch
assert torch.cuda.is_available(), 'Set Runtime -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

REPO = 'https://github.com/YOURUSER/pspe.git'   # <-- edit
import os
if not os.path.exists('pspe'):
    !git clone $REPO pspe_repo && mv pspe_repo/* . 2>/dev/null; ls


In [ ]:
# Colab already ships a CUDA-matched torch + HF stack. Do NOT reinstall or pin
# those — it drags the resolver into incompatible torch/torchao/CUDA wheels
# (the classic "incompatible version of torchao" error).
#
# 1) Drop torchao: newer transformers refuses an old torchao, and we don't use
#    it (4-bit goes through bitsandbytes). Removing it sidesteps the clash.
!pip uninstall -q -y torchao
# 2) Light deps Colab may lack — all >= floors it already satisfies elsewhere,
#    so torch / numpy / transformers are left untouched.
!pip install -q -r requirements-notebook.txt
# 3) The package + its HF extra via >= floors (no reinstalls of the base stack).
!pip install -q -e ".[llm]"

# Verify nothing got clobbered: torch still imports and sees the GPU, and
# transformers imports without the torchao complaint.
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '| transformers', transformers.__version__)
!python scripts/check_env.py | tail -20

## 2. Sanity: EuroSAT loads and RGB→NDVI is a real inverse problem

NDVI needs the NIR band, which is dropped from the RGB input — so the encoder
genuinely has to reconstruct a physical quantity the image does not contain.

In [ ]:
from pspe.perceive.dataset import PerceptionDataConfig, PerceptionDataset, collate

data_cfg = PerceptionDataConfig(source='eurosat', n_samples=1024, image_size=64)
train = PerceptionDataset(data_cfg, 'train')
val   = PerceptionDataset(data_cfg, 'val')
print(f'{len(train)} train / {len(val)} val patches')

import matplotlib.pyplot as plt
img, field, cap = train[0]
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img.permute(1, 2, 0)); ax[0].set_title('Sentinel-2 RGB (input)')
ax[1].imshow(field[0], cmap='RdYlGn', vmin=-1, vmax=1); ax[1].set_title('NDVI field (target)')
for a in ax: a.axis('off')
print('caption:', cap)

## 3. Perceive with a real vision backbone (SigLIP), LoRA-only

The backbone stays frozen; only LoRA adapters train. The encoder resizes our
64px imagery to the tower's native resolution internally, so `image_size=64`
is fine. `assert_lora_only` refuses to proceed if any backbone weight is
trainable; the summary is watermarked `backbone_is_stub: false`.

In [ ]:
from pspe.perceive import PerceiveConfig, PerceiveTrainConfig, PerceiveTrainer
from pspe.utils import RunLogger

model_cfg = PerceiveConfig(
    backbone='google/siglip-base-patch16-224',  # or openai/clip-vit-base-patch32, facebook/dinov2-small
    image_size=64, out_grid=64, out_channels=1,  # NDVI is one channel
    lora_r=8, lora_alpha=16, quant='none',        # SigLIP-base is small; bf16 fits the T4
)
train_cfg = PerceiveTrainConfig(epochs=8, batch=16, lr=1e-3, freeze_encoder=True)
trainer = PerceiveTrainer(model_cfg, data_cfg, train_cfg,
                          RunLogger('runs/perceive_eurosat'), device='cuda')
summary = trainer.train()
print('backbone_is_stub:', summary['backbone_is_stub'])   # must be False
print('held-out NDVI rel L2:', round(summary['val/loss/regression'], 4))
print('trainable fraction:', round(summary['params/trainable_fraction'] * 100, 2), '%')
assert summary['backbone_is_stub'] is False

## 4. The two missing perception baselines (Section 7.2)

- **CNN/ViT regression**: the in-tree `tiny` encoder, regression term only
  (contrastive weight 0) — a supervised baseline with no pretrained backbone
  and no alignment.
- **Zero-shot vision**: the frozen SigLIP backbone with no adapters trained
  (0 epochs) — what the pretrained tower reconstructs with no task adaptation.

All three report field-reconstruction error on the same held-out EuroSAT split.

In [ ]:
rows = [('PSPE (SigLIP+LoRA)', summary['val/loss/regression'], summary['backbone_is_stub'])]

# CNN/ViT regression baseline: stub encoder, contrastive off.
cnn = PerceiveTrainer(
    PerceiveConfig(backbone='tiny', image_size=64, out_grid=64, out_channels=1),
    data_cfg,
    PerceiveTrainConfig(epochs=8, batch=16, w_contrastive=0.0, freeze_encoder=False),
    RunLogger('runs/perceive_cnn'), device='cuda')
rows.append(('CNN/ViT regression', cnn.train()['val/loss/regression'], True))

# Zero-shot: real backbone, no adapter training (0 epochs).
zs = PerceiveTrainer(model_cfg, data_cfg,
                     PerceiveTrainConfig(epochs=0, batch=16, freeze_encoder=True),
                     RunLogger('runs/perceive_zeroshot'), device='cuda')
rows.append(('Zero-shot SigLIP', zs.train()['val/loss/regression'], False))

print(f"{'method':<22}{'NDVI rel L2':>12}{'stub':>7}")
for name, err, stub in rows:
    print(f'{name:<22}{err:>12.4f}{str(stub):>7}')

## 5. Explain with a real LM (Qwen2.5-1.5B), trained-in vs post-hoc

Qwen2.5 is a standard causal LM (no packed-patch issue), so it runs directly.
Needs a **trained planner checkpoint** — the next cell builds one, then we
compare trained-in faithfulness against a post-hoc baseline (same LM,
faithfulness loss off) — the control Section 7.2 asks for.

In [ ]:
!python scripts/generate_data.py testbed=dar data.n_trajectories=256
!python scripts/train_simulate.py testbed=dar train.epochs=20 device=cuda
!python scripts/train_plan.py testbed=dar planner.iterations=200 device=cuda

In [ ]:
import torch
from pspe.envs import make_env
from pspe.explain import ExplainConfig, ExplainTrainConfig, ExplainTrainer
from pspe.plan import GaussianFieldPolicy
from pspe.utils import RunLogger

env = make_env('dar', dynamics='truth', grid=64, horizon=12, device='cuda', batched=True)
policy = GaussianFieldPolicy(env.obs_shape[0], env.action_dim).to('cuda')
policy.load_state_dict(torch.load('runs/plan/policy.pt')['policy'])

lm = ExplainConfig(backbone='Qwen/Qwen2.5-1.5B-Instruct', quant='4bit')

def run(tag, use_faith):
    tr = ExplainTrainer(env, policy, lm,
        ExplainTrainConfig(iterations=200, use_faithfulness=use_faith, log_dir=f'runs/{tag}'),
        RunLogger(f'runs/{tag}'), device='cuda')
    s = tr.train()
    return s['eval/faithfulness'], s['backbone_is_stub']

f_trained, stub = run('explain_trained', True)    # trained-in (Eq. 11)
f_posthoc, _    = run('explain_posthoc', False)   # post-hoc control (TalkToAgent-style)
print(f'trained-in F(b) = {f_trained:.3f}   post-hoc F(b) = {f_posthoc:.3f}   stub={stub}')
assert stub is False

## 6. Save results back

Copy `runs/` to Drive so the numbers survive the session. The human study
(`eval/human_rating.py` on `runs/explain_trained/briefs.jsonl`) runs offline
with volunteers — no GPU needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pspe_runs && cp -r runs/* /content/drive/MyDrive/pspe_runs/
print('saved to Drive/pspe_runs')